# EXP-13: Barnase–Barstar Association Kinetics
**Electrostatic Steering Validation**

- **PDB:** 1BRS (2.0 Å) | Chains A (barnase) + D (barstar)
- **Temperature:** 300 K
- **Pipeline:** Prep → Min → Eq → US/WHAM (WT) → US/WHAM (charge-neutralized) → k_on
- **Expected:** k_on ≈ 3.7×10⁸ M⁻¹s⁻¹; electrostatic steering ≥2 orders of magnitude
- **PASS:** k_on in [10⁷, 10¹⁰] AND steering ≥2 orders

In [ ]:
# Install OpenMM + all scientific dependencies (pip — OpenCL platform)
!pip install openmm mdtraj 'pymbar==4.0.3' mdanalysis
!pip install netCDF4 git+https://github.com/choderalab/openmmtools.git

# mpiplus stub — not on PyPI; minimal shim for imports
import os as _os, sys as _sys
_sp = _os.path.join(_sys.prefix, 'lib',
    f'python{_sys.version_info.major}.{_sys.version_info.minor}',
    'site-packages', 'mpiplus')
_os.makedirs(_sp, exist_ok=True)
with open(_os.path.join(_sp, '__init__.py'), 'w') as _f:
    _f.write(
        'class delayed_termination:\n'
        '    def __init__(self, func=None):\n'
        '        self._func = func\n'
        '    def __enter__(self): return self\n'
        '    def __exit__(self, *a): pass\n'
        '    def __call__(self, *args, **kwargs):\n'
        '        if self._func is not None: return self._func(*args, **kwargs)\n'
        '        if len(args) == 1 and callable(args[0]): return args[0]\n'
        '        return self\n'
        'def get_mpicomm(): return None\n'
        'def run_single_node(rank=0, broadcast_result=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def on_single_node(rank=0, broadcast_result=False, sync_nodes=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def distribute(func, seq, *args, send_results_to=None, sync_nodes=False, group_nodes=None):\n'
        '    results = [func(item, *args) for item in seq]\n'
        '    return list(zip(*results)) if results and isinstance(results[0], tuple) else (results, list(seq))\n'
    )
for _k in list(_sys.modules):
    if 'mpiplus' in _k:
        del _sys.modules[_k]

!pip install -q scipy matplotlib pandas requests gemmi

# Verify critical imports
import importlib
all_ok = True
for pkg in ['openmm', 'openmmtools', 'mdtraj', 'pymbar', 'MDAnalysis']:
    try:
        m = importlib.import_module(pkg)
        print(f"\u2713 {pkg} {getattr(m, '__version__', '')}")
    except ImportError as e:
        print(f"\u2717 {pkg}: {e}")
        all_ok = False

if all_ok:
    print("\n\u2705 All packages installed successfully!")
else:
    raise RuntimeError("Package installation failed \u2014 check error messages above")

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive', force_remount=True)
import sys, os, shutil, json, zipfile, logging
from pathlib import Path
import numpy as np

PIPELINE_ROOT = Path('/content/drive/MyDrive/spink7_pipeline')
assert PIPELINE_ROOT.exists()
sys.path.insert(0, str(PIPELINE_ROOT))

EXP_ID = 'EXP-13'
DRIVE_OUTPUT = Path(f'/content/drive/MyDrive/v3_gpu_results/{EXP_ID}')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path(f'/content/{EXP_ID}_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s %(message)s')

# ── Console log capture ──────────────────────────────────────
import io as _io
_log_buf = _io.StringIO()
_orig_stdout_write = sys.stdout.write
_orig_stderr_write = sys.stderr.write
def _stdout_log_write(data):
    _orig_stdout_write(data)
    _log_buf.write(data)
def _stderr_log_write(data):
    _orig_stderr_write(data)
    _log_buf.write(data)
sys.stdout.write = _stdout_log_write
sys.stderr.write = _stderr_log_write
# ─────────────────────────────────────────────────────────────

In [ ]:
# Checkpoint & auto-save utilities
import json, shutil, threading, time as _time
from pathlib import Path

class ExperimentCheckpoint:
    """Drive-backed phase checkpoint for resilient experiment execution."""

    def __init__(self, drive_output: Path, exp_id: str):
        self.drive_output = drive_output
        self.exp_id = exp_id
        self.path = drive_output / 'experiment_checkpoint.json'
        self.state = self._load()

    def _load(self) -> dict:
        if self.path.exists():
            with open(self.path) as f:
                return json.load(f)
        return {'exp_id': self.exp_id, 'phases': {}}

    def save(self):
        self.drive_output.mkdir(parents=True, exist_ok=True)
        with open(self.path, 'w') as f:
            json.dump(self.state, f, indent=2, default=str)

    def is_done(self, phase: str) -> bool:
        return phase in self.state['phases']

    def mark_done(self, phase: str, data: dict = None):
        self.state['phases'][phase] = {
            'timestamp': _time.strftime('%Y-%m-%d %H:%M:%S'),
            'data': data or {},
        }
        self.save()

    def get_data(self, phase: str) -> dict:
        return self.state['phases'].get(phase, {}).get('data', {})

    def summary(self):
        n = len(self.state['phases'])
        print(f'Checkpoint: {self.exp_id} — {n} phase(s) completed')
        for phase, info in self.state['phases'].items():
            print(f'  ✓ {phase} ({info["timestamp"]})')

def start_periodic_sync(work_dir: Path, drive_output: Path, interval_s: int = 300):
    """Background thread that syncs checkpoint/manifest files to Drive every N seconds."""
    stop_event = threading.Event()
    sync_patterns = ['*.chk', '*.json', '*manifest*', '*.npz', '*.npy']
    def _sync():
        while not stop_event.is_set():
            stop_event.wait(interval_s)
            if stop_event.is_set():
                break
            for pat in sync_patterns:
                for f in work_dir.rglob(pat):
                    if f.is_file() and f.stat().st_size < 500_000_000:
                        dest = drive_output / f.relative_to(work_dir)
                        dest.parent.mkdir(parents=True, exist_ok=True)
                        try:
                            shutil.copy2(f, dest)
                        except Exception:
                            pass
    t = threading.Thread(target=_sync, daemon=True, name='drive-sync')
    t.start()
    return stop_event

def restore_from_drive(drive_output: Path, work_dir: Path):
    """On session restart, copy checkpoint/manifest files from Drive back to local."""
    restored = 0
    for pat in ['*.chk', '*manifest*', '*.json']:
        for f in drive_output.rglob(pat):
            if f.is_file():
                dest = work_dir / f.relative_to(drive_output)
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists():
                    shutil.copy2(f, dest)
                    restored += 1
    if restored:
        print(f'Restored {restored} checkpoint/manifest files from Drive')

# Initialize
ckpt = ExperimentCheckpoint(DRIVE_OUTPUT, EXP_ID)
restore_from_drive(DRIVE_OUTPUT, WORK_DIR)
sync_stop = start_periodic_sync(WORK_DIR, DRIVE_OUTPUT, interval_s=300)
ckpt.summary()
print('Auto-save: checkpoint/manifest files sync to Drive every 5 min')

In [ ]:
import openmm
from openmm import unit, Platform
Platform.getPlatformByName('CUDA')
print(f'OpenMM {openmm.__version__}, CUDA ready')

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.config import (
    SystemConfig, MinimizationConfig, EquilibrationConfig,
    ProductionConfig, UmbrellaConfig, WHAMConfig, BOLTZMANN_KJ, KCAL_TO_KJ
)
from src.prep.pdb_fetch import fetch_pdb
from src.prep.pdb_clean import clean_structure
from src.prep.topology import build_topology
from src.prep.solvate import solvate_system
from src.simulate.minimizer import minimize_energy
from src.simulate.equilibrate import run_nvt, run_npt
from src.simulate.umbrella import run_umbrella_campaign, generate_window_centers, compute_overlap_matrix
from src.simulate.platform import select_platform
from src.analyze.wham import solve_wham, bootstrap_pmf_uncertainty
print('Imports OK')

In [ ]:
PDB_ID = '1BRS'
CHAIN_BARNASE = 'A'
CHAIN_BARSTAR = 'D'
TEMPERATURE_K = 300.0  # Barnase-barstar literature uses 300 K
D_REL = 10e-6  # cm^2/s relative diffusion coefficient

system_config = SystemConfig()
min_config = MinimizationConfig(max_iterations=10000)
eq_config = EquilibrationConfig(nvt_duration_ps=500.0, npt_duration_ps=1000.0, temperature_k=TEMPERATURE_K)
umbrella_config = UmbrellaConfig(
    xi_min_nm=1.5, xi_max_nm=4.0, window_spacing_nm=0.05,
    spring_constant_kj_mol_nm2=1000.0, per_window_duration_ns=10.0,
)
wham_config = WHAMConfig(tolerance=1e-6, max_iterations=100000, n_bootstrap=200)

for d in ['prep', 'simulation', 'umbrella_wt', 'umbrella_neutral', 'analysis', 'figures']:
    (WORK_DIR / d).mkdir(parents=True, exist_ok=True)
PREP_DIR = WORK_DIR / 'prep'
SIM_DIR = WORK_DIR / 'simulation'
US_WT_DIR = WORK_DIR / 'umbrella_wt'
US_NEU_DIR = WORK_DIR / 'umbrella_neutral'
ANALYSIS_DIR = WORK_DIR / 'analysis'
FIGURES_DIR = WORK_DIR / 'figures'
print(f'{len(generate_window_centers(umbrella_config))} windows per campaign')

In [ ]:
# Fetch and clean PDB
raw_pdb = fetch_pdb(PDB_ID, output_dir=PREP_DIR)
cleaned_pdb = clean_structure(raw_pdb, chains_to_keep={CHAIN_BARNASE, CHAIN_BARSTAR},
                        remove_heteroatoms=True, remove_waters=True)
with open(cleaned_pdb) as f:
    chains_found = set(l[21] for l in f if l.startswith('ATOM'))
print(f'Chains: {sorted(chains_found)}')

In [ ]:
from openmm.app import PME, ForceField, Simulation, HBonds
from openmm import LangevinMiddleIntegrator, XmlSerializer

topology, system, modeller = build_topology(
    cleaned_pdb, system_config)

# Identify pull groups from chains
pull_group_1, pull_group_2 = [], []
for chain in topology.chains():
    indices = [a.index for a in chain.atoms()]
    if chain.id == CHAIN_BARNASE:
        pull_group_1 = indices
    elif chain.id == CHAIN_BARSTAR:
        pull_group_2 = indices
print(f'Barnase: {len(pull_group_1)} atoms, Barstar: {len(pull_group_2)} atoms')

modeller, n_water, n_pos, n_neg = solvate_system(modeller, system_config)
print(f'Solvated: {n_water} waters')

ff = ForceField(system_config.force_field, system_config.water_model)
system = ff.createSystem(modeller.topology, nonbondedMethod=PME,
    nonbondedCutoff=1.0*unit.nanometer, constraints=HBonds, rigidWater=True)
integrator = LangevinMiddleIntegrator(TEMPERATURE_K*unit.kelvin, 1.0/unit.picosecond, 0.002*unit.picoseconds)
simulation = Simulation(modeller.topology, system, integrator, select_platform('CUDA'))
simulation.context.setPositions(modeller.positions)

min_result = minimize_energy(simulation, min_config)
system_xml_path = SIM_DIR / 'system.xml'
system_xml_path.write_text(XmlSerializer.serialize(system), encoding='utf-8')
print(f'Minimized: {min_result["final_energy_kj_mol"]:.0f} kJ/mol')

In [ ]:
nvt_result = run_nvt(simulation, eq_config, SIM_DIR / 'nvt')
npt_result = run_npt(simulation, eq_config, SIM_DIR / 'npt')
print(f'NPT: T={npt_result["avg_temperature_k"]:.1f} K, rho={npt_result["avg_density_g_cm3"]:.4f}')
eq_state_path = SIM_DIR / 'npt' / 'npt_final_state.xml'
topology_pdb_path = npt_result['topology_path']

for f in [system_xml_path, eq_state_path, Path(topology_pdb_path)]:
    if f.exists(): shutil.copy2(f, DRIVE_OUTPUT / f.name)

In [ ]:
# US Campaign 1: Wild-type
if ckpt.is_done('umbrella_wt'):
    print('⏭ WT umbrella sampling already completed, skipping')
else:
    print('Starting WT umbrella sampling...')
    us_wt = run_umbrella_campaign(
        equilibrated_state_path=eq_state_path, system_xml_path=system_xml_path,
        config=umbrella_config, pull_group_1=pull_group_1, pull_group_2=pull_group_2,
        output_dir=US_WT_DIR, topology_pdb_path=Path(topology_pdb_path),
        platform_name='CUDA', n_workers=1,
    )
    print(f'WT: {len(us_wt)} windows')
    ckpt.mark_done('umbrella_wt', {'n_windows': len(us_wt)})

In [ ]:
# Create charge-neutralized system: zero all protein partial charges
system_neutral = XmlSerializer.deserialize(system_xml_path.read_text())

# Find NonbondedForce and zero charges on all protein atoms
protein_indices = set(pull_group_1 + pull_group_2)
for force in system_neutral.getForces():
    if isinstance(force, openmm.NonbondedForce):
        for idx in protein_indices:
            charge, sigma, epsilon = force.getParticleParameters(idx)
            force.setParticleParameters(idx, 0.0 * unit.elementary_charge, sigma, epsilon)
        # Also update exceptions involving protein atoms
        for exc_idx in range(force.getNumExceptions()):
            i, j, chargeProd, sig, eps = force.getExceptionParameters(exc_idx)
            if i in protein_indices or j in protein_indices:
                force.setExceptionParameters(exc_idx, i, j, 0.0 * chargeProd.unit, sig, eps)
        break

neutral_xml_path = SIM_DIR / 'system_neutral.xml'
neutral_xml_path.write_text(XmlSerializer.serialize(system_neutral), encoding='utf-8')
print('Charge-neutralized system created')

In [ ]:
# US Campaign 2: Charge-neutralized
if ckpt.is_done('umbrella_neutral'):
    print('⏭ Charge-neutralized umbrella sampling already completed, skipping')
else:
    print('Starting charge-neutralized umbrella sampling...')
    us_neutral = run_umbrella_campaign(
        equilibrated_state_path=eq_state_path, system_xml_path=neutral_xml_path,
        config=umbrella_config, pull_group_1=pull_group_1, pull_group_2=pull_group_2,
        output_dir=US_NEU_DIR, topology_pdb_path=Path(topology_pdb_path),
        platform_name='CUDA', n_workers=1,
    )
    print(f'Neutral: {len(us_neutral)} windows')
    ckpt.mark_done('umbrella_neutral', {'n_windows': len(us_neutral)})

In [ ]:
# WHAM for both campaigns
def extract_pmf(us_results, label):
    xi_ts = [r['xi_timeseries'] for r in us_results]
    wc = np.array([r['window_center_nm'] for r in us_results])
    ks = np.full(len(us_results), umbrella_config.spring_constant)
    ov = compute_overlap_matrix(xi_ts)
    min_ov = min(ov[i,i+1] for i in range(len(us_results)-1))
    print(f'{label} min adjacent overlap: {min_ov:.3f}')
    wham = solve_wham(xi_ts, wc, ks, TEMPERATURE_K, wham_config)
    bs = bootstrap_pmf_uncertainty(xi_ts, wc, ks, TEMPERATURE_K, wham_config)
    return wham, bs, min_ov

wham_wt, bs_wt, ov_wt = extract_pmf(us_wt, 'WT')
wham_neutral, bs_neutral, ov_neutral = extract_pmf(us_neutral, 'Neutral')

In [ ]:
# Compute k_on using Szabo-Shoup-Northrup method
# k_on = 4*pi*D_rel * integral(exp(beta*W(r)) / r^2, r_bound to r_max)^(-1)
beta = 1.0 / (BOLTZMANN_KJ * TEMPERATURE_K)

def compute_kon(wham_result, D_rel_cm2_s, label):
    xi = wham_result['xi_bins']  # nm
    pmf_kj = wham_result['pmf_kj_mol']
    finite = np.isfinite(pmf_kj)
    xi_f = xi[finite]
    pmf_f = pmf_kj[finite]
    
    # Convert xi from nm to cm for integration
    xi_cm = xi_f * 1e-7  # nm -> cm
    
    # Integrand: exp(beta*W(r)) / r^2
    integrand = np.exp(beta * pmf_f) / (xi_cm ** 2)
    
    # Trapezoidal integration
    integral = np.trapz(integrand, xi_cm)
    
    # k_on = 4*pi*D_rel / integral (units: cm^3/mol/s -> M^-1 s^-1 via * N_A * 1e-3)
    k_on_cm3_per_s = 4.0 * np.pi * D_rel_cm2_s / integral
    k_on_M_s = k_on_cm3_per_s * 6.022e23 * 1e-3  # cm^3/s -> L/mol/s = M^-1 s^-1
    
    print(f'{label}: k_on = {k_on_M_s:.2e} M^-1 s^-1')
    return k_on_M_s

k_on_wt = compute_kon(wham_wt, D_REL, 'WT')
k_on_neutral = compute_kon(wham_neutral, D_REL, 'Neutral')

steering_ratio = k_on_wt / k_on_neutral if k_on_neutral > 0 else float('inf')
steering_orders = np.log10(steering_ratio) if steering_ratio > 0 else 0
print(f'\nElectrostatic steering ratio: {steering_ratio:.1f}x ({steering_orders:.1f} orders)')

In [ ]:
# Success classification
EXPECTED_KON = 3.7e8
kon_pass = 1e7 <= k_on_wt <= 1e10
steering_pass = steering_orders >= 2.0

if kon_pass and steering_pass:
    verdict = 'PASS'
elif (1e6 <= k_on_wt <= 1e11) or steering_orders >= 1.0:
    verdict = 'MARGINAL'
else:
    verdict = 'FAIL'

print(f'k_on = {k_on_wt:.2e} (expected {EXPECTED_KON:.2e})')
print(f'Steering = {steering_orders:.1f} orders (need >= 2)')
print(f'Verdict: {verdict}')

In [ ]:
# Figures
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# PMF comparison
xi_wt = wham_wt['xi_bins']; pmf_wt = wham_wt['pmf_kcal_mol']
xi_n = wham_neutral['xi_bins']; pmf_n = wham_neutral['pmf_kcal_mol']
fw = np.isfinite(pmf_wt); fn = np.isfinite(pmf_n)
ax1.plot(xi_wt[fw], pmf_wt[fw], 'b-', lw=2, label='Wild-type')
ax1.plot(xi_n[fn], pmf_n[fn], 'r--', lw=2, label='Charge-neutralized')
ax1.set_xlabel('\u03be (nm)'); ax1.set_ylabel('PMF (kcal/mol)')
ax1.set_title('PMF: WT vs Charge-Neutralized')
ax1.legend(); ax1.grid(True, alpha=0.3)

# k_on bar chart
ax2.bar(['WT', 'Neutral'], [np.log10(k_on_wt), np.log10(k_on_neutral)],
        color=['blue', 'red'], alpha=0.7)
ax2.axhline(np.log10(EXPECTED_KON), color='green', ls='--', label=f'Exp. {EXPECTED_KON:.1e}')
ax2.set_ylabel('log\u2081\u2080(k_on / M\u207b\u00b9s\u207b\u00b9)')
ax2.set_title(f'Electrostatic Steering: {steering_orders:.1f} orders')
ax2.legend(); ax2.grid(True, alpha=0.3)

fig.suptitle('EXP-13: Barnase\u2013Barstar Kinetics', fontsize=16)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'kinetics_analysis.png', dpi=150)
plt.close(fig)
print('Figures saved')

In [ ]:
results = {
    'experiment_id': EXP_ID, 'pdb_id': PDB_ID, 'temperature_k': TEMPERATURE_K,
    'k_on_wt_M_s': k_on_wt, 'k_on_neutral_M_s': k_on_neutral,
    'steering_ratio': steering_ratio, 'steering_orders': steering_orders,
    'expected_k_on': EXPECTED_KON, 'verdict': verdict,
    'wham_wt_converged': wham_wt['converged'],
    'wham_neutral_converged': wham_neutral['converged'],
    'min_overlap_wt': float(ov_wt), 'min_overlap_neutral': float(ov_neutral),
    'npt_avg_density': npt_result['avg_density_g_cm3'],
}
with open(WORK_DIR / 'results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)

np.savez(ANALYSIS_DIR / 'pmf_wt.npz', xi_bins=wham_wt['xi_bins'],
         pmf_kj=wham_wt['pmf_kj_mol'], pmf_kcal=wham_wt['pmf_kcal_mol'])
np.savez(ANALYSIS_DIR / 'pmf_neutral.npz', xi_bins=wham_neutral['xi_bins'],
         pmf_kj=wham_neutral['pmf_kj_mol'], pmf_kcal=wham_neutral['pmf_kcal_mol'])
print('Results saved')

In [ ]:
sync_stop.set()  # Stop periodic sync before final copy

for item in WORK_DIR.rglob('*'):
    if item.is_file():
        dest = DRIVE_OUTPUT / item.relative_to(WORK_DIR)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, dest)

ckpt.mark_done('complete')

# ── Save full console log ────────────────────────────────────
_console_log_text = _log_buf.getvalue()
(WORK_DIR / 'console_log.txt').write_text(_console_log_text)
(DRIVE_OUTPUT / 'console_log.txt').write_text(_console_log_text)
print(f'Console log saved: {len(_console_log_text)} chars')
# ─────────────────────────────────────────────────────────────
zip_path = Path(f'/content/{EXP_ID}_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for item in WORK_DIR.rglob('*'):
        if item.is_file(): zf.write(item, f'{EXP_ID}/{item.relative_to(WORK_DIR)}')
print(f'Zip: {zip_path.stat().st_size/1e6:.1f} MB')
files.download(str(zip_path))